In [1]:
# Cell 1: load the clean table and put every sensor on the same 15-minute grid
from pathlib import Path
import pandas as pd

clean = pd.read_csv(Path("../data/processed/strawberry_clean.csv"), parse_dates=["time"])
usable = clean[(clean["phase"] == "transit") & ~clean["partial_sensor"] & ~clean["handling_event"]]

# Mean temp per sensor per 15-minute block, so a sensor logging every 5 min doesn't outweigh one logging every 10
sensor_columns = ["shipment", "pallet", "layer"]
blocks = (
    usable.set_index("time")
    .groupby(sensor_columns)["temp_c"]
    .resample("15min")
    .mean()
    .dropna()  # blocks with no readings (gaps) come out as NaN
    .reset_index()
)

first_block = blocks.groupby(sensor_columns)["time"].transform("min")
blocks["days_in_transit"] = (blocks["time"] - first_block) / pd.Timedelta(days=1)

print(f"Rows before resampling: {len(usable)}")
print(f"Rows after resampling:  {len(blocks)}")
print("\nRows (15-min blocks) per shipment:")
print(blocks.groupby("shipment").size().to_string())

Rows before resampling: 32047
Rows after resampling:  21226

Rows (15-min blocks) per shipment:
shipment
1    2943
2    1148
3    5028
4    4013
5    5265
6    2829


In [2]:
# Cell 2: three models, each tested on a truck it has never seen (leave-one-shipment-out)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


def make_features(data, use_days):
    """One-hot pallet columns (Front is the baseline, so it has no column), plus optional days_in_transit."""
    features = pd.DataFrame({
        "pallet_Middle": (data["pallet"] == "Middle").astype(float),
        "pallet_Rear": (data["pallet"] == "Rear").astype(float),
    })
    if use_days:
        features["days_in_transit"] = data["days_in_transit"]
    return features


results = []
for held_out in sorted(blocks["shipment"].unique()):
    train = blocks[blocks["shipment"] != held_out]
    test = blocks[blocks["shipment"] == held_out]

    # M0: always predict the training mean
    m0_pred = [train["temp_c"].mean()] * len(test)

    # M1: pallet only
    m1 = LinearRegression().fit(make_features(train, use_days=False), train["temp_c"])
    m1_pred = m1.predict(make_features(test, use_days=False))

    # M2: pallet + days in transit
    m2 = LinearRegression().fit(make_features(train, use_days=True), train["temp_c"])
    m2_pred = m2.predict(make_features(test, use_days=True))

    results.append({
        "held_out": held_out,
        "M0": mean_absolute_error(test["temp_c"], m0_pred),
        "M1": mean_absolute_error(test["temp_c"], m1_pred),
        "M2": mean_absolute_error(test["temp_c"], m2_pred),
    })

mae = pd.DataFrame(results).set_index("held_out")
m1_wins = (mae["M1"] < mae["M0"]).sum()
m2_wins = (mae["M2"] < mae["M1"]).sum()

report = mae.round(3).astype(object)
report.loc["mean"] = mae.mean().round(3)
report.loc["wins"] = ["-", f"{m1_wins}/6 vs M0", f"{m2_wins}/6 vs M1"]
print("MAE on the held-out shipment (°C):")
print(report.to_string())

MAE on the held-out shipment (°C):
             M0         M1         M2
held_out                             
1         0.707      0.624      0.717
2         0.722      0.683       0.61
3         0.866      0.853      0.832
4         1.231      1.287      1.272
5         0.655       0.57      0.525
6         0.772      0.793      0.773
mean      0.825      0.802      0.788
wins          -  4/6 vs M0  5/6 vs M1


In [3]:
# Cell 3: pick the model (more detail only if it wins on at least 4 of 6 unseen trucks), fit on all data
chosen = "M0"
if m1_wins >= 4:
    chosen = "M1"
    if m2_wins >= 4:
        chosen = "M2"
print(f"Chosen model: {chosen}  (M1 beat M0 in {m1_wins}/6, M2 beat M1 in {m2_wins}/6)\n")

if chosen == "M0":
    print(f"Base temperature: {blocks['temp_c'].mean():.2f} °C (same for every pallet, no drift)")
else:
    use_days = chosen == "M2"
    final = LinearRegression().fit(make_features(blocks, use_days), blocks["temp_c"])
    coef = dict(zip(make_features(blocks, use_days).columns, final.coef_))

    pallet_effect = {"Front": 0.0, "Middle": coef["pallet_Middle"], "Rear": coef["pallet_Rear"]}
    average_effect = sum(pallet_effect.values()) / 3

    print(f"Base temperature (Front pallet{', day 0' if use_days else ''}): {final.intercept_:.2f} °C")
    for pallet, effect in pallet_effect.items():
        # also shown relative to the average of the three pallets, to compare with Step 8
        print(f"  {pallet:<6} vs Front: {effect:+.2f} °C   vs pallet average: {effect - average_effect:+.2f} °C")
    if use_days:
        print(f"Drift: {coef['days_in_transit']:+.3f} °C per day")

Chosen model: M2  (M1 beat M0 in 4/6, M2 beat M1 in 5/6)

Base temperature (Front pallet, day 0): 1.83 °C
  Front  vs Front: +0.00 °C   vs pallet average: -0.10 °C
  Middle vs Front: -0.13 °C   vs pallet average: -0.23 °C
  Rear   vs Front: +0.43 °C   vs pallet average: +0.33 °C
Drift: +0.099 °C per day


In [ ]:
# Cell 4 (Step 9c): do daytime or layer improve M2 on unseen trucks?

# Candidate features on the 15-minute blocks (logged clock time, time zone unknown)
blocks["is_daytime"] = blocks["time"].dt.hour.between(8, 17).astype(float)  # 8:00-17:59
blocks["layer_Middle"] = (blocks["layer"] == "Middle").astype(float)  # Bottom is the baseline
blocks["layer_Top"] = (blocks["layer"] == "Top").astype(float)

# Quick check before modeling: is daytime warmer than night within each truck?
day_night = blocks.groupby(["shipment", "is_daytime"])["temp_c"].mean().unstack()
day_night.columns = ["night_c", "day_c"]
day_night["day_minus_night"] = day_night["day_c"] - day_night["night_c"]
print(day_night.round(2).to_string())
print(f"Daytime warmer in {(day_night['day_minus_night'] > 0).sum()} of 6 shipments\n")

EXTRA_COLUMNS = {"is_daytime": ["is_daytime"], "layer": ["layer_Middle", "layer_Top"]}


def make_features_extra(data, extras):
    """M2's features (pallet + days_in_transit) plus the columns of any extra feature names."""
    features = make_features(data, use_days=True)
    for name in extras:
        for column in EXTRA_COLUMNS[name]:
            features[column] = data[column].values
    return features


def loso_mae(extras):
    """Leave-one-shipment-out MAE (°C) per held-out shipment, for M2 + extras."""
    errors = {}
    for held_out in sorted(blocks["shipment"].unique()):
        train = blocks[blocks["shipment"] != held_out]
        test = blocks[blocks["shipment"] == held_out]
        model = LinearRegression().fit(make_features_extra(train, extras), train["temp_c"])
        errors[held_out] = mean_absolute_error(test["temp_c"], model.predict(make_features_extra(test, extras)))
    return pd.Series(errors)


candidates = {"M2": [], "M4": ["is_daytime"], "M5": ["layer"]}
mae_extra = pd.DataFrame({name: loso_mae(extras) for name, extras in candidates.items()})
mae_extra.index.name = "held_out"
wins = {name: (mae_extra[name] < mae_extra["M2"]).sum() for name in ["M4", "M5"]}

report = mae_extra.round(3).astype(object)
report.loc["mean"] = mae_extra.mean().round(3)
report.loc["wins vs M2"] = ["-", f"{wins['M4']}/6", f"{wins['M5']}/6"]
print("MAE on the held-out shipment (°C):")
print(report.to_string())

# Same rule as before: keep a feature only if it beats M2 in at least 4 of 6
kept = [extras[0] for name, extras in candidates.items() if name != "M2" and wins[name] >= 4]
print(f"\nFeatures kept: {kept if kept else 'none - M2 stays'}")
if len(kept) > 1:
    print("Note: both passed on their own; the combined model was not tested separately")

final = LinearRegression().fit(make_features_extra(blocks, kept), blocks["temp_c"])
coef = dict(zip(make_features_extra(blocks, kept).columns, final.coef_))
baseline = "Front pallet, day 0" + (", Bottom layer" if "layer" in kept else "") + (", night" if "is_daytime" in kept else "")
print(f"\nBase temperature ({baseline}): {final.intercept_:.2f} °C")
for column, value in coef.items():
    unit = "°C per day" if column == "days_in_transit" else "°C"
    print(f"  {column:<16} {value:+.3f} {unit}")